# 01 — EDA (from `bike_sharing_sagemaker.ipynb`)

Uses `../data/raw/hour.csv`. Install dev deps: `pip install -r ../requirements-dev.txt`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
hour_df = pd.read_csv("../data/raw/hour.csv", parse_dates=["dteday"])
# day_df = pd.read_csv('day.csv')

In [ ]:
hour_df.head()

In [ ]:
print("Hour DataFrame shape:", hour_df.shape)

In [ ]:
hour_df.columns

In [ ]:
# convert the date 
hour_df["dteday"] = pd.to_datetime(hour_df["dteday"])

In [ ]:
hour_df.info()

In [ ]:
# Check missing values
hour_df.isnull().sum()

In [ ]:
# Check for the duplicates in the dataset
hour_df.duplicated().sum()

In [ ]:
hour_df.describe()

In [ ]:
print("Dataset loaded successfully")

print("Rows:", hour_df.shape[0])
print("Columns:", hour_df.shape[1])

print("\nMissing Values")
print(hour_df.isnull().sum())

print("\nDuplicate Rows:", hour_df.duplicated().sum())

# EDA

In [ ]:
plt.figure(figsize=(10,6))
sns.set_style("whitegrid")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(hour_df['cnt'], bins = 50, kde = True)
plt.title('Distribution of Bike Rentals (cnt)')
plt.xlabel('Count')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Insights - the data is right skewed, with most of the bike rentals being on the lower end of the count spectrum. There are some outliers with very high counts, which may indicate peak rental times or special events. The distribution suggests that while most rentals are relatively low, there are occasional spikes in demand.

# Here mean is misleading as the data is right skewed and it would overestimate the data. Here median would be a better measure of the central tendency of the data. 

In [ ]:
# Check Demand by hr
plt.figure(figsize=(12, 6))
sns.lineplot(x ='hr', y ='cnt', data = hour_df)
plt.title("Average Bike Demand by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Count")
plt.show()

In [ ]:
# Insights from the distribution of bike rentals:
# - Morning Peak around 8-9 AM, likely due to work time or school commutes.
# - Evening Peak around 4-6 PM, also likely due to students commuting for the classes.

In [ ]:
# Demand by weekday
plt.figure(figsize=(10,6))
sns.barplot(x='weekday', y='cnt', data=hour_df)
plt.title('Demand for Bike Rentals by Weekday')
plt.xlabel('Weekday (0=Sunday, 6=Saturday)')
plt.ylabel('Average Count')
plt.show()

In [ ]:
# Insights:
# - Weekday demand is generally higher than weekend demand, with peaks on weekdays likely due to work and school commutes.
# - Sunday (0) has the lowest demand, while weekdays (1-5) show higher demand, with a slight dip on Saturday (6).

In [ ]:
# Demand over time
daily_trend = hour_df.groupby('dteday')['cnt'].mean()

plt.figure(figsize=(14,6))
daily_trend.plot()
plt.title('Daily Average Bike Rentals Over Time')
plt.xlabel('Date')  
plt.ylabel('Average Count')
plt.show()

In [ ]:
# Insights :
# - Trend is relatively stable with some fluctuations, indicating consistent demand for bike rentals over time. There are occasional peaks and troughs, which may correspond to seasonal variations, holidays, or special events affecting bike rental usage.

# Analysis - It is going up during the summer season and then it is going down during the winter season.

In [ ]:
# Demand by Season
season_labels = ['Spring', 'Summer', 'Fall', 'Winter']
plt.Figure(figsize=(10,6))
sns.barplot(x='season', y = 'cnt', data = hour_df)
plt.title('Demand by Season')
plt.xticks(range(len(season_labels)), season_labels)
plt.show()

In [ ]:
# Insights:
# - Peak Seasons - Summer and Fall
# - Low Season - Spring

In [ ]:
# Demand by month
plt.Figure(figsize=(12,6))
sns.boxplot(x = 'mnth', y = 'cnt', data = hour_df)
plt.title('Demand by Month')
plt.show()

In [ ]:
# Insights: 
# - We have a lot of outlier in the months 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12. This is because of the seasonality in the data. We can see that the demand is higher in the summer months (6, 7, 8) and lower in the winter months (1, 2, 3). The outliers could be due to special events or holidays that cause a spike in demand.

In [ ]:
plt.Figure(figsize=(12,6))
labels = ['Clear', 'Mist', 'Light Snow/Rain', 'Heavy Rain/Snow']
sns.barplot(x = 'weathersit', y = 'cnt', data = hour_df)
plt.title('Demand by Weather Situation')
plt.xlabel('Weather Situation')
plt.ylabel('Average Count')
plt.xticks(range(len(labels)), labels)
plt.show()

In [ ]:
# Insights :
# - Highest Demand in Clear and Mist
# - Lowest Demand in Heavy Rain/Snow and Light Snow/Rain

In [ ]:
# Temperature vs Demand
plt.figure(figsize=(10,6))
sns.scatterplot(x='temp', y='cnt', data=hour_df, alpha=0.3)
plt.title("Temperature vs Demand")
plt.show()

In [ ]:
# Insights:

# - With increase in temperature, the demand increases. This is likely because warmer weather encourages more people to rent bikes, while colder temperatures may deter them from doing so.

# - But Higher the humidity, the lower the demand showing peak summer season and winter season have lower demand then others.

# - Positive correlation between temerature and Cnt.

In [ ]:
# Correlation Analysis All features
plt.figure(figsize=(14,8))
sns.heatmap(hour_df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Correlation Analysis
numeric_cols = ['temp', 'atemp', 'hum', 'windspeed', 'cnt']
plt.figure(figsize=(10,6))
sns.heatmap(hour_df[numeric_cols].corr(method='pearson'), annot=True, cmap='coolwarm', fmt=".2f", square=True)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
from scipy.stats.contingency import association

categorical_cols = ['season', 'yr', 'mnth', 'hr',
    'holiday', 'weekday', 'workingday', 'weathersit']

cat_association = pd.DataFrame(index = categorical_cols, columns = categorical_cols)

# Compare each pair of categorical features with others using Cramér's V

for col1 in categorical_cols:
    for col2 in categorical_cols:
        if col1 == col2:
            cat_association.loc[col1, col2] = 1
        else:
            table = pd.crosstab(hour_df[col1], hour_df[col2])
            score = association(table, method = 'cramer', correction = False)
            cat_association.loc[col1, col2] = score

cat_association = cat_association.astype(float)
plt.figure(figsize=(10,6))
sns.heatmap(cat_association, annot=True, cmap = 'Blues', fmt = '.2f')
plt.title("Categorical Feature Association Heatmap (Cramer's V)")
plt.show()

In [ ]:
for col in categorical_cols:
    print("\n", col)
    print(hour_df.groupby(col)['cnt'].mean().sort_values())

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.barplot(x=col, y='cnt', data=hour_df)
    plt.title(f"{col} vs cnt")
    plt.show()

In [ ]:
# Importance Score for the categorical features
importance = {}
for col in categorical_cols:
    means = hour_df.groupby(col)['cnt'].mean()
    importance[col] =  means.max() - means.min()

importance_df = pd.DataFrame.from_dict(importance, orient='index', columns=['Importance'])
importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df
importance_df.plot(kind='bar', figsize=(10,5), title="Categorical Feature Importance (Mean Difference)")
plt.show()

In [ ]:
# Insights - 
# - Drop casual and registered as High Correlation (leakage!) with cnt.
# - Drop atemp as it is highly correlated with temp.
# - Take hr, season, weathersit, weekday, mnth as categorical features.
# - Take temp, hum, windspeed as numerical features.

#### Demand Drivers

##### - Hour of day → strongest factor

##### - Working day → strong factor

##### - Weather → strong factor

##### - Temperature → strong factor

#### Behavioral Insight

##### - This is a commuter-driven system

##### - Demand is time + weather dependent

##### - Relationships are not purely linear


### STEP 3 — Data Cleaning + Preprocessing